<a href="https://colab.research.google.com/github/mocha-dts/data-science-2026/blob/main/pertemuan3_mochamadrochmatullah_240401010041.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sesi 3 – Data Cleaning: Missing, Outlier & Ekstraksi**

STEP 0 — Load & eksplorasi awal


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

URL = "https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo"
df = pd.read_csv(URL)

df.info()
df.describe()
print('Shape awal:', df.shape)
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
Shape awal: (130, 7)
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


STEP 1 — Hapus Duplikat

In [ ]:
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

Setelah hapus duplikat: (130, 7)


STEP 2 — Normalisasi String

In [ ]:
print('Nilai unik kolom [kota] SEBELUM normalisasi:')
print(df['kota'].unique())
print('\nNilai unik kolom [kondisi] SEBELUM normalisasi:')
print(df['kondisi'].unique())

Nilai unik kolom [kota] SEBELUM normalisasi:
['jogja' 'Medan' 'Depok' 'YGY' 'Jakarta' 'jakarta' 'Yogyakarta' 'Bandung'
 'Surabaya' 'dpk' 'sby' 'Makassar' 'mdn' 'medan' 'Semarang' 'semarang'
 'yogyakarta' 'Jogja' 'JAKARTA' 'Smg' 'DEPOK' 'Bdg' 'makassar' 'surabaya'
 'MAKASSAR' 'depok' 'bandung' 'Bandung ' 'SURABAYA' 'Mksr' ' Jakarta']

Nilai unik kolom [kondisi] SEBELUM normalisasi:
['baik' 'Bagus' 'Sedang' 'baik sekali' 'SEDANG' 'sedang' 'BAIK' 'rusak'
 'cukup' 'Baik' 'Cukup' 'perlu renovasi' 'bagus' 'jelek' 'RUSAK']


In [ ]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

In [ ]:
print('Nilai unik kolom [kota] SETELAH normalisasi:')
print(df['kota'].unique())
print('\nNilai unik kolom [kondisi] SETELAH normalisasi:')
print(df['kondisi'].unique())

Nilai unik kolom [kota] SETELAH normalisasi:
['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']

Nilai unik kolom [kondisi] SETELAH normalisasi:
['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']


STEP 3 — Imputasi Missing Values

In [ ]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
print(f'\nMissing values setelah imputasi: {df.isnull().sum().sum()}')


Missing values setelah imputasi: 0


STEP 4 — Tangani Outlier (IQR Fence)

In [ ]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
 Q1, Q3 = df[col].quantile([0.25, 0.75])
 IQR = Q3 - Q1
 df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

STEP 5 — Validasi & Ekspor

In [ ]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Shape akhir: (130, 7)
Dataset bersih tersimpan!


Akses API JSONPlaceholder dan simpan respons sebagai DataFrame

In [ ]:
api_url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(api_url)

if response.status_code == 200:
    data_json = response.json()
    df_api = pd.DataFrame(data_json)
    print("\nData API JSONPlaceholder berhasil diambil!")
    print(df_api.head())
else:
    print(f"\nGagal mengakses API. Status code: {response.status_code}")


Data API JSONPlaceholder berhasil diambil!
   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   
3       1   4                               eum et est occaecati   
4       1   5                                 nesciunt quas odio   

                                                body  
0  quia et suscipit\nsuscipit recusandae consequu...  
1  est rerum tempore vitae\nsequi sint nihil repr...  
2  et iusto sed quo iure\nvoluptatem occaecati om...  
3  ullam et saepe reiciendis voluptatem adipisci\...  
4  repudiandae veniam quaerat sunt sed\nalias aut...  
